# పాఠం 01 - AI ఏజెంట్ల పరిచయం

**AI ఏజెంట్లు ప్రారంభకుల కోసం** కోర్సులోని మొదటి పాఠంలో స్వాగతం!

**AI ఏజెంట్** అనేది పెద్ద భాషా మోడల్ (LLM) ను reasoning ఇంజన్‌గా ఉపయోగించి వాస్తవ ప్రపంచంలో *చర్యలు* చేపట్టగల ప్రోగ్రామ్ — APIs కాల్ చేయడం, డేటాబేస్‌లను క్వెరీ చేయడం, లేదా కోడ్ నడపడం ద్వారా వినియోగదారుడి కోసం ఒక లక్ష్యాన్ని సాధించడం.

ఈ నోటుబుక్‌లో మీరు మీ మొదటి ఏజెంట్‌ను నిర్మించబోతున్నారు: ఒక **ప్రయాణ ఏజెంట్** ఇది సెలవు గములను సూచిస్తుంది. ఈ ప్రయాణంలో మీరు ఇలా నేర్చుకుంటారు:

1. **Microsoft Agent Framework** ఉపయోగించి Azure AI Foundry ఏజెంట్ సర్వీస్‌కు కనెక్ట్ అవడం.
2. ఏజెంట్‌కు ఒక **టూల్** ఇవ్వడం — ఇది పరోక్ష Python ఫంక్షన్ అంతే.
3. ఏజెంట్‌ను నడపడం మరియు దాని ప్రతిస్పందనను పరిశీలించడం.
4. ఏజెంట్ ప్రతిస్పందన టోకెన్-ద్వారా-టోకెన్ స్ట్రీమ్ చేయడం.


## సెటప్

ఈ నోట్‌బుక్‌ను నడిపించే ముందు, మీరు ఈ క్రింది వాటిని కలిగి ఉన్నారని నిర్ధారించుకోండి:

1. **డిప్లాయ్ చేసిన చాట్ మోడల్ ఉన్న Azure AI Foundry ప్రాజెక్ట్** (ఉదాహరణకు `gpt-4o-mini`).
2. **Azure CLI లో లాగిన్ అయి ఉండాలి** — మీ టెర్మినల్‌లో `az login` ను నడపండి.
3. **అవసరమైన ఎంట్విరాన్మెంట్ వేరియబల్స్ సెట్ చేయండి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ Azure AI Foundry ప్రాజెక్ట్ ఎండ్‌పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీ డిప్లాయ్ చేసిన మోడల్ పేరు.

క్రింద ఉన్న సెల్ మీరు అవసరమయ్యే Python ప్యాకేజీలను ఇనిస్టాల్ చేస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## మీ మొదటి ఏజెంట్‌ను సృష్టించడం

ఏజెంట్‌కు రెండు విషయాలు అవసరం:

- **సూచనలు** అవి *అది ఎవరు* మరియు *ఎలా ప్రవర్తించాలి* అంటూ చెప్పతాయి (ఒక సిస్టమ్ ప్రాంప్ట్).
- **పరికరాలు** — ఏజెంట్ సమాచారం గణించడానికి లేదా చర్యలు చేపట్టడానికి పిలవగల `@tool` తో అలంకరించబడిన Python ఫంక్షన్లు.

తదుపరి ఒక సులభమైన పరికరాన్ని నిర్వచిస్తాము, అది ప్రసిద్ధ సెలవు గమ్యస్థలాల జాబితాను అందిస్తుంది. యూజర్ ప్రయాణ సలహాలు కోరినప్పుడు ఏజెంట్ ఈ పరికరాన్ని ఉపయోగిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## స్ట్రీమింగ్ ప్రతిస్పందనలు

మరి ఇంటరాక్టివ్ అనుభవం కోసం మీరు ఏజెంట్ యొక్క ప్రతిస్పందనను **స్ట్రీమ్** చేయవచ్చు. పూర్తి సమాధానం కోసం వేచి ఉండే బదులు, ఏజెంట్ తయారుచేసే టెక్స్ట్ భాగాలను వరుసగా ఇస్తుంది. ఇది రియల్ టైమ్ లో అవుట్‌పుట్ చూపించాలనుకునే చాట్ ఇంటర్‌ఫేస్‌లలో ప్రత్యేకంగా ఉపయోగకరం.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

ఈ పాఠంలో మీరు నేర్చుకున్నది:

- **AzureAIProjectAgentProvider** ద్వారా Azure AI Foundry Agent Service కి కనెక్ట్ అయ్యే ప్రొవైడర్కు **సృష్టించండి**.
- ఏజెంట్ మీ Python ఫంక్షన్లను పిలవడానికి `@tool` డెకరేటర్ ఉపయోగించి **టూల్ ని నిర్వచించండి**.
- యూజర్ సందేశంతో ఏజెంట్ను **రన్ చేసి** దాని ప్రతిస్పందనను ప్రింట్ చేయండి.
- రియల్-టైమ్ అవుట్పుట్ కోసం **ప్రతిస్పందనలను స్ట్రీమ్ చేయండి**.

తరువాతి పాఠంలో మేము ఏజెంటిక్ ఫ్రేమ్‌వర్క్‌లను మరింత లోతుగా పరిశీలించి ఏజెంట్లకు శక్తివంతమైన టూల్స్ మరియు బహు-దశల తర్క సామర్థ్యాలను ఎలా ఇవ్వాలో నేర్చుకుంటాము.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మనం ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో తప్పులు లేదా అసత్యతలు ఉండవచ్చు అని గమనించండి. అసలు పత్రం వారి స్వదేశ భాషలో ఉన్నది అధికారిక మూలం olarak పరిగణించవలెను. ముఖ్యమైన సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదం సిఫార్సు చేయబడుతుంది. ఈ అనువాదం వలన కలిగే ఏవైనా అపార్థాలు లేదా తప్పు గ్రహింపులకు మేము బాధ్యులే కాదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
